# 01 - Basics: The Robot Loop

## What / Why / How

**What we are trying to do:** Build the smallest complete robot loop: state, observation, action, dynamics, and feedback.

**Why this matters:** Every robot, from a toy rover to a humanoid, repeatedly senses, decides, and acts. If this loop is unclear, advanced control and learning will feel like magic.

**How we will do it:** Simulate a 1D robot moving to a target, inspect the feedback variables, then change gains and limits to see how behavior changes.

## Goal

Build the basic mental model of a robot:

- A robot has state: where it is, how fast it moves, what it senses.
- A robot receives observations from sensors.
- A controller or policy chooses actions.
- Actuators apply actions to the physical world.
- Feedback closes the loop.

In learning-based robotics, a policy is often trained from data. In classical robotics, a controller is usually designed from equations. You will use both.

In [ ]:
import math
import random
from collections import defaultdict

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see the plot: pip install -r requirements.txt")

## The Robot Loop

At each time step:

1. Sense the world.
2. Estimate the state.
3. Decide an action.
4. Apply the action.
5. Repeat.

We will start with the smallest possible robot: a 1D point that moves toward a target.

In [ ]:
dt = 0.05
target = 1.0
x = -1.0
velocity_limit = 1.2
k_p = 1.8

history = []
for step in range(120):
    t = step * dt
    observation = x
    error = target - observation
    action_velocity = np.clip(k_p * error, -velocity_limit, velocity_limit)
    x = x + action_velocity * dt
    history.append((t, observation, error, action_velocity))

history = np.array(history)
print("final position:", round(float(x), 4))
print("final error:", round(float(target - x), 4))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(8, 3))
    plt.plot(history[:, 0], history[:, 1], label="position")
    plt.axhline(target, color="black", linestyle="--", label="target")
    plt.xlabel("time [s]")
    plt.ylabel("x")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    plot_unavailable()
    print("first rows: time, x, error, action")
    print(history[:5])

## State, Observation, Action, Policy

A useful vocabulary:

- State: the real variables needed to describe the robot and world.
- Observation: what sensors report. It may be noisy or incomplete.
- Action: the command sent to actuators.
- Policy: a function that maps observations to actions.
- Dynamics: how the world changes after an action.

A classical controller may be hand-designed. A learned policy is trained from examples or rewards.

In [ ]:
def proportional_policy(observation, target, gain=1.0, limit=1.0):
    error = target - observation
    return np.clip(gain * error, -limit, limit)

for gain in [0.5, 1.0, 2.0, 5.0]:
    x = -1.0
    for _ in range(80):
        u = proportional_policy(x, target=1.0, gain=gain, limit=1.0)
        x += u * dt
    print(f"gain={gain:>3}: final x={x:.3f}, error={1.0 - x:.3f}")

## Exercises

1. Change `velocity_limit`. What happens when the robot is too weak?
2. Change `k_p`. What happens when gain is low or high?
3. Add sensor noise: `observation = x + np.random.normal(0, 0.02)`.
4. Explain why feedback is more robust than sending one fixed command.